<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Promptable segmentation with SAM 2

**[Promptable segmentation with SAM 2](https://learning.nextia-ai.com/courses/vision/m05/promptable-segmentation-with-sam-2/)** · Vision AI: See, Count and Inspect · Module 5, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** get a mask from a click or a box with SAM 2, with no training; score the masks against exact masks; see why a box prompt is safer than a point; cut products out of their photos; and catch and fix a bad cut-out with a simple rule.

| | |
|---|---|
| **Time** | About 40 minutes. The cells run in about a minute after the setup. |
| **Needs** | An internet connection for the setup cells and the model (SAM 2.1 tiny, about 150 MB). No account and no key. |
| **You should know** | Masks, semantic and instance, from [Semantic and instance segmentation](https://learning.nextia-ai.com/courses/vision/m05/semantic-and-instance-segmentation/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 with PyTorch 2.15.0 and Transformers 5.22.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m05/promptable-segmentation-with-sam-2/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M05-L02-promptable-segmentation-with-sam-2/promptable-segmentation-with-sam-2-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, torch 2.15.0, torchvision 0.29.2, timm 1.0.31, transformers 5.22.2, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time. PyTorch is large (about 200 MB to download): the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2 torch==2.15.0 torchvision==0.29.2 timm==1.0.31 transformers==5.22.2
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)
import torch; print("torch", torch.__version__)
import torchvision; print("torchvision", torchvision.__version__)
import timm; print("timm", timm.__version__)
import transformers; print("transformers", transformers.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 18 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/catalogue.zip": "83752b3ce15a1b99f3b9201cc979769d7abe39633754a84865d7e60f0b3d7814",
    "archives/pallets.zip": "cec4b211fa4e434936fc0c1980accfc74392f1ced9de6c04b5421b9ec6af56e3",
    "compare.py": "e9794acacbce00c240906cc8b3e2256b8505ff4a8074b7507a8b159cd39b4826",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Load SAM 2

**SAM 2** (Segment Anything Model 2, by Meta) draws a mask for an object that you point at. You give it a **prompt**: a point on the object, or a box around it. It needs no training on your photos.

The next cell downloads the smallest version, SAM 2.1 tiny (about 150 MB, Apache-2.0), and defines `segment`: a photo and prompts in, one mask per prompt out.

You may see a message that the checkpoint is of type `sam2_video`. That is expected: the file also holds the parts for video, and we use its parts for single photos.

In [ ]:
import torch
from transformers import Sam2Model, Sam2Processor
torch.set_num_threads(4)
# SAM 2.1 tiny (Meta, Apache-2.0), at a fixed revision: about 150 MB the first time.
NAME, REV = "facebook/sam2.1-hiera-tiny", "de431c4043854a71d8101e17995dfe596bf101a5"
processor = Sam2Processor.from_pretrained(NAME, revision=REV)
sam = Sam2Model.from_pretrained(NAME, revision=REV).eval()

def segment(img, boxes=None, points=None):
    """One mask per prompt. boxes: [[x0, y0, x1, y1], ...]; points: [[x, y], ...] (one click per object)."""
    kw = {}
    if boxes is not None:
        kw["input_boxes"] = [boxes]
    if points is not None:
        kw["input_points"] = [[[p] for p in points]]
        kw["input_labels"] = [[[1] for _ in points]]
    x = processor(images=img, return_tensors="pt", **kw)
    with torch.no_grad():
        out = sam(**x, multimask_output=False)
    masks = processor.post_process_masks(out.pred_masks, x["original_sizes"])[0]
    return masks[:, 0].numpy() > 0

print(f"SAM 2.1 tiny: {sum(p.numel() for p in sam.parameters()) / 1e6:.1f} million numbers")

## 2. Box prompts on a tight pallet

S-063 is a tight pallet: 22 boxes that touch. We know the exact mask of every box, so we can score SAM's masks. Here, each true box is the prompt. In real work, the box comes from a person or from a detector.

In [ ]:
from PIL import ImageDraw
pallets = json.load(open("data/pallets/test.json"))

def polygon_mask(polygon, w, h):
    m = Image.new("L", (w, h), 0)
    ImageDraw.Draw(m).polygon([(polygon[i], polygon[i + 1]) for i in range(0, len(polygon), 2)], fill=1)
    return np.asarray(m, bool)

scene = next(im for im in pallets["images"] if im["scene_id"] == "S-063")
anns = [a for a in pallets["annotations"] if a["image_id"] == scene["id"]]
truth = [polygon_mask(a["segmentation"][0], scene["width"], scene["height"]) for a in anns]
photo = Image.open("data/" + scene["file_name"]).convert("RGB")
# a box prompt: [x0, y0, x1, y1]; the course's boxes are [x, y, w, h]
boxes = [[a["bbox"][0], a["bbox"][1], a["bbox"][0] + a["bbox"][2], a["bbox"][1] + a["bbox"][3]] for a in anns]

import time
start = time.perf_counter()
box_masks = segment(photo, boxes=boxes)
print(f"{len(box_masks)} masks in {time.perf_counter() - start:.1f} s")
box_iou = [cp.mask_iou(m, t) for m, t in zip(box_masks, truth)]
print(f"mean mask IoU {np.mean(box_iou):.3f}; masks below 0.8: {sum(v < 0.8 for v in box_iou)}")

> **Check.** You should see a mean mask IoU of 0.929, and no mask below 0.8. Mask IoU is the IoU of [Boxes and IoU](https://learning.nextia-ai.com/courses/vision/m04/boxes-and-iou/), counted in pixels: the pixels in both masks, divided by the pixels in either mask.

## 3. One click per box

A click is faster than a box. Here, the click is the centre of each box, a little below the middle (60% down): the front face.

> **Predict.** Will one click per box give better or worse masks than a box? Which boxes will be the worst?

In [ ]:
points = [[a["bbox"][0] + a["bbox"][2] / 2, a["bbox"][1] + a["bbox"][3] * 0.6] for a in anns]
point_masks = segment(photo, points=points)
point_iou = [cp.mask_iou(m, t) for m, t in zip(point_masks, truth)]
print(f"mean mask IoU {np.mean(point_iou):.3f}; masks below 0.8: {sum(v < 0.8 for v in point_iou)}")

worst = int(np.argmin(point_iou))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, m, title in zip(axes, (truth[worst], point_masks[worst], box_masks[worst]),
                        ("true mask", f"one click: IoU {point_iou[worst]:.2f}", f"box: IoU {box_iou[worst]:.2f}")):
    ax.imshow(photo); ax.imshow(np.ma.masked_where(~m, m), alpha=0.6, cmap="autumn"); ax.set_title(title); ax.axis("off")
axes[1].plot(*points[worst], "o", color="lime", markersize=8)
plt.show()

> **Check.** You should see a mean of 0.881, with 2 masks below 0.8; the worst has an IoU of 0.73. A click on the front face gives the front face: SAM cannot know that the lid on top belongs to the same box. A box says where the object ends.

## 4. Cut products out of their photos

The catalogue team wants product photos on a plain background. Each catalogue photo was cut from an Open Images photo around a box drawn by a person. Here are four of those boxes, in the pixels of our 256 x 256 photos.

A cut-out can go wrong without any error message. So we check each one with a simple rule (our rule, not a standard): the box around the mask should match the labelled box. When their IoU is under 0.8, a person looks at it.

In [ ]:
BOXES = {
    "P-9e093eaa": {
        "label": "bowl",
        "box": [
            24.8,
            45.0,
            231.2,
            211.0
        ]
    },
    "P-b0286c43": {
        "label": "lamp",
        "box": [
            15.5,
            0,
            244.3,
            256
        ]
    },
    "P-9ed21416": {
        "label": "mug",
        "box": [
            34.1,
            0,
            255.5,
            256
        ]
    },
    "P-53ba97e3": {
        "label": "plant_pot",
        "box": [
            17.0,
            64.1,
            239.0,
            246.9
        ]
    }
}

def mask_box(m):
    """The box around the True pixels of a mask, as [x, y, w, h]."""
    ys, xs = np.where(m)
    return [int(xs.min()), int(ys.min()), int(xs.max() - xs.min() + 1), int(ys.max() - ys.min() + 1)]

def box_check(m, box):
    """Our rule: the IoU of the mask's own box with the labelled box. Under 0.8: the cut-out needs a fix."""
    return cp.iou(mask_box(m), [box[0], box[1], box[2] - box[0], box[3] - box[1]])

masks, photos = {}, {}
for pid, b in BOXES.items():
    photos[pid] = Image.open(f"data/catalogue/test/{b['label']}/{pid}.jpg").convert("RGB")
    masks[pid] = segment(photos[pid], boxes=[b["box"]])[0]
    print(pid, b["label"], f"check {box_check(masks[pid], b['box']):.3f}")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, pid in zip(axes, BOXES):
    cut = np.where(masks[pid][..., None], np.asarray(photos[pid]), 255)
    ax.imshow(cut); ax.set_title(f"{pid}: {box_check(masks[pid], BOXES[pid]['box']):.2f}"); ax.axis("off")
plt.show()

> **Check.** Three photos pass with checks above 0.94. P-53ba97e3 (a basket of eggs and flowers, labelled "plant pot") has a check of 0.715: it needs a fix. The authors of the photos are in `data/catalogue/manifest.csv` (CC BY 2.0).

In [ ]:
from scipy import ndimage
bad = masks["P-53ba97e3"]
pieces, n = ndimage.label(bad)
print("separate pieces in the mask:", n)
x, y, w, h = mask_box(bad)
print("the mask's box:", [x, y, w, h], "| the labelled box:", [round(v) for v in BOXES["P-53ba97e3"]["box"]])

The mask is in 142 separate pieces. Most are tiny: single leaves and petals outside the basket. One small piece at the top edge makes the mask's box much taller than the labelled box.

## 5. Your turn: fix the cut-out

> **Your turn.** Keep only the **largest** piece of the mask of P-53ba97e3. Then compute the box check of the new mask, and put it in `fixed_check`.
>
> Hint: `pieces` holds a number for each piece (0 is the background). `ndimage.sum(bad, pieces, range(1, n + 1))` gives the size of each piece.

In [ ]:
fixed_check = ...

In [ ]:
expect('fixed_check', fixed_check, 0.919, tolerance=0.005)

## 6. What it costs

SAM 2.1 tiny needs no labels and no training. It takes about a second per pallet photo on a laptop CPU, because it reads the photo once and then answers each prompt quickly. It still needs a prompt for every object: a box from a detector or a person, or a click. And it can be wrong in quiet ways, so a cheap check such as the box rule is part of the method.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/vision/m05/promptable-segmentation-with-sam-2/) has the pictures, the comparison with the course's full run and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Measure from masks](https://learning.nextia-ai.com/courses/vision/m05/measure-from-masks/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m05/promptable-segmentation-with-sam-2/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.